# reASITIC demo notebook

This notebook walks through the typical design flow for a planar
spiral inductor: load tech, build geometry, analyse, optimise, plot.

Prerequisites: ``pip install reASITIC[plot]`` (matplotlib for plotting).

In [ ]:
import sys
from pathlib import Path
ROOT = Path('.').resolve().parents[0]
sys.path.insert(0, str(ROOT / 'src'))
import reasitic
from reasitic.report import design_report
from reasitic.network import linear_freqs, two_port_sweep, write_touchstone_file
from reasitic.optimise import optimise_square_spiral
from reasitic.plot import plot_shape, plot_sweep, plot_lr_matrix

## 1. Load the BiCMOS tech file

In [ ]:
tech = reasitic.parse_tech_file(ROOT.parent / 'run' / 'tek' / 'BiCMOS.tek')
for m in tech.metals:
    print(f'{m.name}: t={m.t} μm, rsh={m.rsh:.4f} Ω/sq, layer={m.layer}')

## 2. Build a 3-turn 200 μm spiral on m3

In [ ]:
sp = reasitic.square_spiral(
    'L1', length=200, width=10, spacing=2, turns=3,
    tech=tech, metal='m3',
)
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
plot_shape(sp, ax=ax)
plt.show()

## 3. Multi-frequency design report

In [ ]:
rpt = design_report(sp, tech, freqs_ghz=[1.0, 2.4, 5.0, 10.0])
print(rpt.format_text())

## 4. Frequency sweep + plot S parameters

In [ ]:
fs = linear_freqs(0.1, 10.0, 0.1)
sweep = two_port_sweep(sp, tech, fs)
import numpy as np
fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
ax[0].plot(fs, [abs(s[0,0]) for s in sweep.S], label='|S11|')
ax[0].plot(fs, [abs(s[1,0]) for s in sweep.S], label='|S21|')
ax[0].set_ylabel('|S|')
ax[0].legend()
Ls = [pi.L_nH for pi in sweep.pi]
ax[1].plot(fs, Ls, 'b-', label='L (nH)')
ax[1].set_xlabel('Frequency (GHz)')
ax[1].set_ylabel('L (nH)')
plt.tight_layout()
plt.show()

## 5. Optimise for a 2 nH inductor

In [ ]:
res = optimise_square_spiral(
    tech, target_L_nH=2.0, freq_ghz=2.4, metal='m3',
)
print(f'Optimal: L={res.length_um:.0f} μm, W={res.width_um:.1f}, S={res.spacing_um:.1f}, N={res.turns:.2f}')
print(f'Achieved: L={res.L_nH:.3f} nH, Q={res.Q:.1f}')

## 6. Export to Touchstone for SPICE simulation

In [ ]:
out = ROOT / 'examples' / 'L1_demo.s2p'
write_touchstone_file(out, sweep.to_touchstone_points(param='S'))
print(f'Wrote {len(fs)} freq points to {out}')

## 7. Heatmap of the partial-L matrix

In [ ]:
fig, ax = plt.subplots()
plot_lr_matrix(sp, ax=ax)
plt.show()